<div style="display: flex; background-color: RGB(255,114,0);" >
<h1 style="margin: auto; padding: 30px; ">ANALYSE DU STOCK ET DES VENTES DU SITE BOTTLENECK</h1>
</div>

# OBJECTIF DE CE NOTEBOOK

Bienvenue dans l'outil plébiscité par les analystes de données Jupyter.

Il s'agit d'un outil permettant de mixer et d'alterner codes, textes et graphique.

Cet outil est formidable pour plusieurs raisons:

+ il permet de tester des lignes de codes au fur et à mesure de votre rédaction, de constater immédiatement le résultat d'un instruction, de la corriger si nécessaire.
+ De rédiger du texte pour expliquer l'approche suivie ou les résultats d'une analyse et de le mettre en forme grâce à du code html ou plus simple avec **Markdown**
+ d'agrémenter de graphiques

Pour vous aider dans vos premiers pas à l'usage de Jupyter et de Python, nous avons rédigé ce notebook en vous indiquant les instructions à suivre.

Il vous suffit pour cela de saisir le code Python répondant à l'instruction donnée.

Vous verrez de temps à autre le code Python répondant à une instruction donnée mais cela est fait pour vous aider à comprendre la nature du travail qui vous est demandée.

Et garder à l'esprit, qu'il n'y a pas de solution unique pour résoudre un problème et qu'il y a autant de résolutions de problèmes que de développeurs ;)...



<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 1 - Importation des librairies et chargement des fichiers</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">1.1 - Importation des librairies</h3>
</div>

In [1]:
#Importation de la librairie Pandas
import pandas as pd

In [2]:
#Importation de la librairie plotly express
import plotly.express as px

In [3]:
#Importation d'autres librairies utiles au projet
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
#Gestion des avertissements
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [5]:
#Trouver dans Google l'instruction permettant d'afficher toutes les colonnes d'un dataframe
#Saisir, dans Google, les mots clés "display all columns dataframe Pandas", par exemple.
#Dans les résultats de la recherche, privilégiez les solutions provenants de Stack Overflow ou Medium

# Configuration pour afficher toutes les colonnes
pd.set_option('display.max_columns', None)
# Configuration pour afficher toutes les lignes
pd.set_option('display.max_rows', None)
# Configuration pour la largeur d'affichage
pd.set_option('display.width', None)

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">1.2 - Chargement des fichiers</h3>
</div>

In [6]:
#Importation du fichier web.xlsx
df_web = pd.read_excel("web.xlsx")
#Importation du fichier erp.xlsx
df_erp = pd.read_excel("erp.xlsx")
#importation du fichier liaison.xlsx
df_liaison = pd.read_excel("liaison.xlsx")

FileNotFoundError: [Errno 2] No such file or directory: 'web.xlsx'

In [ ]:
# Affichage des premières lignes de chaque DataFrame
print("Aperçu des données web :")
print(df_web.head())
print("\nAperçu des données erp :")
print(df_erp.head())
print("\nAperçu des données liaison :")
print(df_liaison.head())

<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 2 - Analyse exploratoire des fichiers</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1 - Analyse exploratoire du fichier erp.xlsx</h3>
</div>

In [ ]:
#Afficher les dimensions du dataset
print("Le tableau comporte {} observation(s) ou article(s)".format(df_erp.shape[0]))
print("Le tableau comporte {} colonne(s)".format(df_erp.shape[1]))

In [ ]:
#Consulter le nombre de colonnes
#La nature des données dans chacune des colonnes
#Le nombre de valeurs présentes dans chacune des colonnes
print("Information sur les colonnes du dataset :")
print(df_erp.info())

In [ ]:
#Vérifier si il y a les lignes en doublons dans la colonne product_id
nb_doublons = df_erp['product_id'].duplicated().sum()
print(f"Nombre de doublons dans la colonne product_id : {nb_doublons}")
if nb_doublons > 0:
    print("\nLignes en doublon :")
    print(df_erp[df_erp['product_id'].duplicated(keep=False)])

In [ ]:
#Afficher les 5 premières lignes de la table
print("Les 5 premières lignes du dataset :")
print(df_erp.head())

Il y a visiblement des **incohérences** entre stock_quantity et stock_status

In [ ]:
#Afficher les valeurs distinctes de la colonne stock_status
print("Valeurs uniques dans stock_status :")
print(df_erp['stock_status'].unique())
#À quelle(s) autre(s) colonne(s) sont-elles liées ?
print("\nRelation entre stock_status et stock_quantity :")
print(df_erp.groupby('stock_status')['stock_quantity'].describe())

**Interprétations** :

Il y a une **forte corrélation** entre stock_status et stock_quantity, mais certaines incohérences existent :
Certains produits "instock" ont **0** unités en stock, ce qui peut être une erreur de mise à jour.
Certains produits "outofstock" ont un **stock négatif** ou légèrement positif, ce qui peut refléter des ajustements d'inventaire.
Il pourrait être utile de vérifier la logique de mise à jour du stock_status pour éviter ces incohérences.

In [ ]:
#Création d'une colonne "stock_status_2
#La valeur de cette deuxième colonne sera fonction de la valeur dans la colonne "stock_quantity"
#si la valeur de la colonne "stock_quantity" est nulle renseigner "outofstock" sinon mettre "instock"
df_erp['stock_status_2'] = np.where(df_erp['stock_quantity'] <= 0, 'outofstock', 'instock')

In [ ]:
#Vérifions que les 2 colonnes sont identiques:
#Les 2 colonnes sont strictement identiques si les valeurs de chaque ligne sont strictement identiques 2 à 2
#La comparaison de 2 colonnes peut se réaliser simplement avec l'instruction ci-dessous:
print("Comparaison ligne par ligne des colonnes stock_status et stock_status_2 :")
print(df_erp["stock_status"] == df_erp["stock_status_2"])
#Le résultat est l'affichage de True ou False pour chacune des lignes du dataset
#C'est un bon début, mais difficile à exploiter

In [ ]:
#Mais il est possible de synthétiser ce résultat en effectuant la somme de cette colonne:
#True vaut 1 et False 0
#Nous devrions obtenir la somme de 825 qui correspond au nombre de lignes dans ce dataset
nb_lignes_identiques = (df_erp["stock_status"] == df_erp["stock_status_2"]).sum()
print(f"Nombre de lignes identiques : {nb_lignes_identiques}")
print(f"Nombre total de lignes : {len(df_erp)}")

In [ ]:
#Si les colonnes ne sont absolument pas identiques ligne à ligne alors identifier la ligne en écart
##Dans ce cas je vous ce lien pour apprendre à réaliser des filtres dans Pandas:
##https://bitbucket.org/hrojas/learn-pandas/src/master/
##Lesson 3
lignes_differentes = df_erp[df_erp["stock_status"] != df_erp["stock_status_2"]]
print("Lignes où stock_status et stock_status_2 diffèrent :")
print(lignes_differentes[['product_id', 'stock_quantity', 'stock_status', 'stock_status_2']])

On a donc bien nos **2** lignes incohérentes

In [ ]:
#Corriger la ou les données incohérentes
mask = (df_erp['stock_quantity'] == 0) & (df_erp['stock_status'] == 'instock')
df_erp.loc[mask, 'stock_status'] = 'outofstock'
#Verification en utilisant le même code que plus haut pour afficher les problemes
print("Vérification après correction :")
print("Nombre de lignes différentes :", (df_erp["stock_status"] != df_erp["stock_status_2"]).sum())

In [ ]:
# Corriger la ou les données incohérentes
mask = (df_erp['stock_quantity'] > 0) & (df_erp['stock_status'] == 'outofstock')
df_erp.loc[mask, 'stock_status'] = 'instock'

# Vérification en utilisant le même code que plus haut pour afficher les problèmes
print("Vérification après correction :")
print("Nombre de lignes différentes :", (df_erp["stock_status"] != df_erp["stock_status_2"]).sum())

Les lignes incohérentes ont été **corrigées**.

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1.1 - Analyse exploratoire de chaque variable du fichier erp.xlsx</h3>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1.1.1 - Analyse de la variable PRIX</h3>
</div>

In [ ]:
###############
## LES PRIX  ##
###############

#Vérification des prix: Y a t-il des prix non renseignés, négatifs ou nuls?
#Afficher le ou les prix non renseignés dans la colonne "price"
print("Nombres d'articles avec un prix non renseigné : {}".format(df_erp['price'].isna().sum()))
#Afficher le prix minimum de la colonne "price"
print("\nPrix minimum : {:.2f}€".format(df_erp['price'].min()))
#Afficher le prix maximum de la colonne "price"
print("Prix maximum : {:.2f}€".format(df_erp['price'].max()))
#Affichier les prix inférieurs à 0 (qu'est ce qu'il faut en faire ?)
prix_negatifs = df_erp[df_erp['price'] < 0]
print("\nArticles avec prix négatifs :")
print(prix_negatifs[['product_id', 'price']] if len(prix_negatifs) > 0 else "Aucun prix négatif trouvé")

In [ ]:
# Analyse détaillée des produits avec prix négatifs
prix_negatifs = df_erp[df_erp['price'] < 0]
print("Détail des produits avec prix négatifs :")
print(prix_negatifs[['product_id', 'price', 'onsale_web', 'purchase_price', 'stock_quantity', 'stock_status']])

# Vérification si ces prix négatifs sont cohérents avec le prix d'achat
print("\nComparaison avec les prix d'achat :")
print(prix_negatifs[['product_id', 'price', 'purchase_price']])

Nous avons maintenant **3** produits (**product_id : 4233, 5017 et 6594**) dont les prix semblent incohérents. Le mieux serait de se référer 
à **un responsable ou à un référent métier** pour savoir s'il est normal d'avoir des prix négatifs
Comme ce n'est ici pas possible, nous allons simplement **les laisser puisqu'ils ne sont de toute façon pas en vente sur le site**.

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1.1.2 - Analyse de la variable STOCK</h3>
</div>

In [ ]:
#######################
### stock_quantity  ###
#######################

#Vérification de la colonne stock quantity
#Afficher la quantité minimum de la colonne "stock_quantity"
print("Quantité minimum en stock : {}".format(df_erp['stock_quantity'].min()))
#Afficher la quantité maximum de la colonne "stock_quantity"
print("Quantité maximum en stock : {}".format(df_erp['stock_quantity'].max()))
#Afficher les stocks inférieurs à 0 (qu'est ce qu'il faut en faire ?)
stocks_negatifs = df_erp[df_erp['stock_quantity'] < 0]
print("\nArticles avec stock négatif :")
print(stocks_negatifs[['product_id', 'stock_quantity']] if len(stocks_negatifs) > 0 else "Aucun stock négatif trouvé")

Ces **2** produits en **stock négatifs** posent problème, là encore le mieux serait de demander à un **référent métier** d'où peut bien venir cette écart

Comme ce n'est pas possible, nous allons **remplacer ces valeurs négatives par des 0**.

In [ ]:
#Modification des valeurs négatives de stock_quantity par 0
df_erp['stock_quantity'] = df_erp['stock_quantity'].apply(lambda x: 0 if x < 0 else x)

In [ ]:
################################
## Valorisation des stocks    ##
################################

#Évaluer la valeur immobilisée en stock
#Identifier les produits qui mobilisent le plus de trésorerie
#Aider à la gestion des approvisionnements


# Calcul de la valeur du stock au prix d'achat
df_erp['valeur_stock_achat'] = df_erp['stock_quantity'] * df_erp['purchase_price']
df_erp['valeur_stock_vente'] = df_erp['stock_quantity'] * df_erp['price']

print("Valorisation du stock :")
print(f"\nValeur totale du stock au prix d'achat : {df_erp['valeur_stock_achat'].sum():,.2f}€")
print(f"Valeur totale du stock au prix de vente : {df_erp['valeur_stock_vente'].sum():,.2f}€")

# Analyse des produits avec le plus de valeur immobilisée
print("\nTop 10 des produits avec la plus grande valeur de stock (prix d'achat) :")
print(df_erp.nlargest(10, 'valeur_stock_achat')[
    ['product_id', 'stock_quantity', 'purchase_price', 'valeur_stock_achat']
])

**Interprétation** :

L’entreprise a près de **300K€** investis dans le stock (coût d’achat), si tout le stock était vendu au prix de vente, il générerait environ **532K€** de revenus.
Cela représente une **marge potentielle de 233 406,13 €** (531 946,20 - 298 540,07).
Une telle analyse peut aider à **identifier l'importance de la gestion des stocks et des coûts d'immobilisation**.
Le top 10 des produits avec la plus grande valeur de stock en prix d'achat montre que **certains articles immobilisent plus de 10 000 € chacun**.
Ces produits peuvent nécessiter une **attention particulière** en termes de gestion du stock et des ventes.
Produits à forte immobilisation : Ceux avec un coût élevé en stock doivent être surveillés pour éviter une surstockage et des risques de dépréciation.

**Opportunité d'optimisation** :

•	Si un produit immobilise beaucoup de trésorerie mais ne se vend pas bien, il peut être intéressant de réduire les achats ou liquider le stock.
•	Inversement, si ces produits se vendent bien, ils doivent être priorisés pour l'approvisionnement.

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1.1.3 - Analyse de la variable ONSALE_WEB</h3>
</div>

In [ ]:
#Vérification de la colonne onsale_web et des valeurs qu'elle contient? Que signifient-elles?
print("Valeurs uniques dans onsale_web :")
print(df_erp['onsale_web'].value_counts())
print("\nPourcentage de chaque valeur :")
print(df_erp['onsale_web'].value_counts(normalize=True).mul(100).round(2), "%")

onsale_web est un indicateur binaire de **disponibilité sur le web**
0 = non disponible à la vente sur le web
1 = disponible à la vente sur le web
Cette information est importante pour comprendre la stratégie de vente en ligne

In [ ]:
#Quelles sont les colonnes à conserver selon vous?
print("Liste des colonnes actuelles :")
for col in df_erp.columns:
    print(f"- {col}")
print("\nToutes les colonnes semblent pertinentes sauf 'stock_status_2' qui est redondante")

In [ ]:
#################################
## Analyses croisées           ##
#################################

#Détecter des incohérences entre le stock et la disponibilité web
#Comprendre la politique de prix selon le statut du stock
#Identifier des opportunités d'optimisation (produits en stock mais non vendus)

# Relation entre onsale_web et stock_status
print("Répartition des produits par status de stock et disponibilité web :")
print(pd.crosstab(df_erp['stock_status'], df_erp['onsale_web'], margins=True))

# Statistiques de prix par status de stock
print("\nStatistiques de prix par status de stock :")
print(df_erp.groupby('stock_status')['price'].describe().round(2))

# Analyse des produits en stock mais non disponibles à la vente
produits_stock_non_vente = df_erp[(df_erp['stock_quantity'] > 0) & (df_erp['onsale_web'] == 0)]
print(f"\nNombre de produits en stock mais non disponibles à la vente : {len(produits_stock_non_vente)}")
if len(produits_stock_non_vente) > 0:
    print("\nExemples de ces produits :")
    print(produits_stock_non_vente[['product_id', 'stock_quantity', 'price', 'onsale_web']].head())

Produits en stock mais non en vente en ligne : **65** produits concernés.
Pourquoi 65 produits sont-ils en stock mais pas en vente ? Bug, politique interne, ou stock non destiné à la vente en ligne ? A vérifier avec **un.e responsable**

Pas de grande différence de prix entre les produits en stock et ceux en rupture.
Pas de politique de prix basée sur le statut du stock (les prix ne changent pas selon la disponibilité).

**Actions à envisager** :

•	Vérifier si ces produits doivent être remis en vente en ligne ou s’ils sont bloqués volontairement.
•	Identifier si ces produits sont en fin de série, réservés à un autre canal (ex. magasins physiques), ou s'il s'agit d'un problème technique.
•	Si ces produits sont immobilisés sans raison, ils représentent un manque à gagner et doivent être réintégrés dans la stratégie de vente en ligne.

In [ ]:
#Supprimer la colonnecomportant le libellé "stock_status_2" car elle est redondante 
#avec la colonne "stock_status".
df_erp = df_erp.drop('stock_status_2', axis=1)
print("Colonnes après suppression :")
print(df_erp.columns.tolist())

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1.1.4 - Analyse de la variable prix d'achat</h3>
</div>

In [ ]:
######################
##   prix d'achat   ##
######################

#Vérification de la colonne purchase_price : 
#Afficher le ou les prix non renseignés dans la colonne "purchase_price"
print("Nombres d'articles avec un prix d'achat non renseigné : {}".format(df_erp['purchase_price'].isna().sum()))
#Afficher le prix minimum de la colonne "purchase_price"
print("\nPrix d'achat minimum : {:.2f}€".format(df_erp['purchase_price'].min()))
#Afficher le prix maximum de la colonne "purchase_price"
print("Prix d'achat maximum : {:.2f}€".format(df_erp['purchase_price'].max()))

#Afficher la distribution des prix d'achat
print("\nRésumé statistique des prix d'achat :")
print(df_erp['purchase_price'].describe())

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.2 - Analyse exploratoire du fichier web.xlsx</h3>
</div>
 

In [ ]:
#Dimension du dataset
#Nombre d'observations
print("Nombre d'observations (lignes) :", df_web.shape[0])
#Nombre de caractéristiques
print("Nombre de caractéristiques (colonnes) :", df_web.shape[1])

In [ ]:
#Consulter le nombre de colonnes
#La nature des données dans chacune des colonnes
#Le nombre de valeurs présentes dans chacune des colonnes
print("Informations sur le dataset web :")
print(df_web.info())

Selon vous, quelles sont les colonnes à conserver ?

On supprime les colonnes suivantes : **"tax_class", "post_content", "post_password", "post_content_filtered", "virtual"**

In [ ]:
#Si vous avez défini des colonnes à supprimer, effectuez l'opération
df_web.drop(columns=["tax_class", "post_content", "post_password", "post_content_filtered", 
                     "virtual"], inplace=True)
#vérifier les dimensions du dataset pour s’assurer que les colonnes ont bien été supprimées
print("Nouvelles dimensions du dataset :", df_web.shape)


In [ ]:
#Visualisation des valeurs de la colonne sku
print("Distribution des valeurs de sku :")
print(df_web['sku'].value_counts().head(10))  # Affiche les 10 premiers
print("\nAperçu des valeurs de sku :")
print(df_web['sku'].head(10))  # Affiche les 10 premiers sku
#Quelles sont les valeurs qui ne semblent pas respecter la régle de codification?
non_numeric_sku = df_web[pd.to_numeric(df_web['sku'], errors='coerce').isna()]['sku']
print("Codes articles non numériques :")
print(non_numeric_sku.value_counts())

In [ ]:
#Si vous avez identifié des codes articles ne respectant pas la régle de codification, consultez-les?
print("Détails des articles avec des sku non numériques :")
print(df_web[df_web['sku'].isin(non_numeric_sku)][['sku', 'post_title', 'product_type']])

In [ ]:
#Identifier les lignes sans code articles
lignes_sans_sku = df_web[df_web['sku'].isna()]
print(f"Nombre de lignes sans code article : {len(lignes_sans_sku)}")
print("\nAperçu des lignes sans code article :")
print(lignes_sans_sku[['post_title', 'product_type', 'post_type']].head())

In [ ]:
#Pour les codes articles identifiés, réalisez une analyse et définissez l'action à entreprendre
print("Analyse des lignes sans code article :")
print("\nTypes de produits concernés :")
print(lignes_sans_sku['product_type'].value_counts())
print("\nTypes de post concernés :")
print(lignes_sans_sku['post_type'].value_counts())

In [ ]:
#La clé pour chaque ligne est-elle unique? ou autrement dit, y a-t-il des doublons?
# Convertir tous les sku en string pour assurer une comparaison cohérente
df_web['sku'] = df_web['sku'].astype(str)

# Filtrer pour exclure les sku nuls ou 'nan'
sku_non_nuls = df_web[df_web['sku'].notna() & (df_web['sku'] != 'nan')]

# Vérifier les doublons sur la colonne sku
doublons_sku = sku_non_nuls[sku_non_nuls['sku'].duplicated(keep=False)]
print(f"Nombre de lignes avec des sku en doublon : {len(doublons_sku)}")

if len(doublons_sku) > 0:
    print("\nDétail des doublons (triés par sku) :")
    # Utiliser reset_index() pour éviter les problèmes d'index
    print(doublons_sku[['sku', 'post_title', 'product_type', 'post_type']]
          .sort_values('sku')
          .reset_index(drop=True))
else:
    print("\nAucun doublon trouvé dans les codes articles.")

On a donc **tous les sku qui apparaissent 2 fois**, ainsi que **2 sku de format différents** (qui apparaissent également 2 fois chacun).
On a également **85 sku NaN**

In [ ]:
#Puisque toutes les lignes sont en doubles avec seulement le post_type de différent, et que nous n'avons besoin que des lignes product, supprimons les 
#lignes avec attachment

# Afficher le nombre de lignes avant suppression
print("Nombre de lignes avant suppression :", len(df_web))

# Identifier et supprimer les lignes avec post_type 'attachment'
df_web = df_web[df_web['post_type'] != 'attachment']

# Afficher le nombre de lignes après suppression
print("Nombre de lignes après suppression :", len(df_web))

# Vérifier s'il reste des doublons sur la colonne sku
doublons_restants = df_web[df_web['sku'].duplicated(keep=False)]
if len(doublons_restants) > 0:
    print("\nDoublons restants :")
    print(doublons_restants[['sku', 'post_title', 'post_type']].sort_values('sku').head())
    print(f"\nNombre de doublons restants : {len(doublons_restants)}")
else:
    print("\nAucun doublon restant dans les SKU")

# Réinitialiser l'index du DataFrame après la suppression
df_web = df_web.reset_index(drop=True)

In [ ]:
#Après suppression des doublons attachment, il nous reste bien les 85 sku en NaN, et en toute logique, il doit également nous rester 2 sku de format différent
#Vérifions
print(df_web[df_web['sku'].isin(non_numeric_sku)][['sku', 'post_title', 'product_type']])

C'est bien ça

Afin de simplifier les jointures à venir, le mieux serait de **standardiser ces 2 sku de formats différents**.

In [ ]:
# Standardiser les SKU spécifiques
df_web.loc[df_web['sku'] == '13127-1', 'sku'] = '131271'
df_web.loc[df_web['sku'] == 'bon-cadeau-25-euros', 'sku'] = '90000'

# Vérification des modifications
print(df_web.loc[[147, 736], ['sku']])

In [ ]:
#Voyons à présent à quoi ressemble notre dataset
print("Informations sur le dataset web :")
print(df_web.info())

In [ ]:
#Les lignes sans code article semble être toutes non renseignés
#Pour s'en assurer réaliser les étapes suivantes:

# S'assurer que NaN est bien reconnu dans la colonne 'sku'
df_web['sku'] = df_web['sku'].replace(["", "NaN", "None", "<NA>", "nan"], np.nan)

#1 - Créer un dataframe avec uniquement les lignes sans code article
df_sans_sku = df_web[df_web['sku'].isna()]
print(f"Nombre de lignes sans SKU identifiées : {len(df_sans_sku)}")

#2 - utiliser la fonction df.info() sur ce nouveau dataframe pour observer le nombre de valeur reseigner dans chacune des colonnes
print("Informations détaillées sur les lignes sans SKU :")
print("==============================================")
print(df_sans_sku.info())

# Afficher tout le DataFrame sous forme de tableau
from IPython.display import display
display(df_sans_sku)


3 - Que constatez-vous?

Le tableau contient principalement des **lignes entièrement vides ou avec des valeurs nulles**, à l'exception de quelques lignes qui semblent correspondre à des produits avec des **valeurs incohérentes** (ex. total_sales négatif).

Les seules colonnes complétées sont downloadable et rating_count, et celles-ci ne nous apportent rien.
 → **À supprimer**, elles n'apportent aucune information utile (**83 lignes**).
 
**2** lignes comportent des **valeurs négatives** dans la colonne total_sales, ces lignes sont en grandes parties complétées, et correspondent bien à des articles mais ne possèdent pas de sku.
 → Vérifier avec un **référent métier** pour savoir s'il s'agit : d'une erreur de calcul, d'annulations ou de remboursements mal gérés, d'un problème d'importation des données, etc.
 → Puisqu'on ne peut pas vérifier la cause ici, nous allons **remettre ces valeurs à 0** afin d'éviter de supprimer les fiches produits

In [ ]:
#On remplace les valeurs négatives par 0
df_web['total_sales'] = df_web['total_sales'].apply(lambda x: max(x, 0))

#On génère 2 nouveaux sku pour ces 2 lignes
df_web.loc[586, 'sku'] = 80000
df_web.loc[588, 'sku'] = 80001

In [ ]:
#On supprime les 83 lignes inutiles
df_web = df_web.dropna(subset=['sku'])

#Et enfin, on vérifie l'unicité
print(df_web['sku'].is_unique)  # Doit renvoyer True si tous les SKU sont uniques
print(df_web['sku'].isna().sum())  # Doit renvoyer 0 si plus aucun NaN
display(df_web)

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.3 - Analyse exploratoire du fichier liaison.xlsx</h3>
</div>

In [ ]:
#Dimension du dataset
#Nombre d'observations
print("Nombre d'observations (lignes) :", df_liaison.shape[0])
#Nombre de caractéristiques
print("Nombre de caractéristiques (colonnes) :", df_liaison.shape[1])

In [ ]:
#Consulter le nombre de colonnes
#La nature des données dans chacune des colonnes
#Le nombre de valeurs présentes dans chacune des colonnes
print("Informations détaillées sur le fichier liaison.xlsx:")
print("==============================================")
print(df_liaison.info())

In [ ]:
#Les valeurs de la colonne "product_id" sont elles toutes uniques?
doublons_product_id = df_liaison[df_liaison['product_id'].duplicated(keep=False)]
print("Nombre de doublons dans product_id :", len(doublons_product_id))

if len(doublons_product_id) > 0:
    print("\nAperçu des product_id en double :")
    print(doublons_product_id.sort_values('product_id'))
else:
    print("\nTous les product_id sont uniques")

In [ ]:
#Les valeurs de la colonne "id_web" sont-elles toutes uniques?
doublons_id_web = df_liaison[df_liaison['id_web'].duplicated(keep=False)]
print("Nombre de doublons dans id_web :", len(doublons_id_web))

if len(doublons_id_web) > 0:
    print("\nAperçu des id_web en double :")
    print(doublons_id_web.sort_values('id_web'))
else:
    print("\nTous les id_web sont uniques")

In [ ]:
#Avons-nous des articles sans correspondance ?

# Vérifier les valeurs nulles dans chaque colonne
print("Articles sans correspondances :")
print("------------------------------")
print("Nombre d'articles sans product_id :", df_liaison['product_id'].isnull().sum())
print("Nombre d'articles sans id_web :", df_liaison['id_web'].isnull().sum())

# Vérifier les lignes avec au moins une valeur manquante
lignes_incompletes = df_liaison[df_liaison.isnull().any(axis=1)]
if len(lignes_incompletes) > 0:
    print("\nAperçu des lignes avec des correspondances manquantes :")
    print(lignes_incompletes)
else:
    print("\nTous les articles ont des correspondances complètes")

Les lignes sans correspondance correpondent en toute logique aux **produits qui ne sont pas mis en vente sur le site**.

In [ ]:
#Puisque vous avions des quelques formats sku différents dans le fichier WEB, vérifions ce qu'il en est ici
non_numeric_id_web = df_liaison[pd.to_numeric(df_liaison['id_web'], errors='coerce').isna()]['id_web']
print("Codes articles non numériques :")
print(non_numeric_id_web.value_counts())

On a effectivement **3 id_web** non numériques, dont 2 que nous avons modifié dans le fichier WEB.

bon-cadeau-25-euros sera donc de nouveau transformé en **90000**, 13127-1 deviendra **131271**, et 14680-1 qui n'était pas dans l'autre fichier deviendra **146801**

In [ ]:
display(df_liaison)

In [ ]:
# Standardiser les id_web spécifiques
df_liaison.loc[df_liaison['id_web'] == '13127-1', 'id_web'] = '131271'
df_liaison.loc[df_liaison['id_web'] == '14680-1', 'id_web'] = '146801'
df_liaison.loc[df_liaison['id_web'] == 'bon-cadeau-25-euros', 'id_web'] = '90000'

# Vérification des modifications
print(df_liaison.loc[[443, 822, 823], ['id_web']])

<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 3 - Jonction des fichiers</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 3.1 - Jonction du fichier df_erp et df_liaison</h3>
</div>

In [ ]:
# Analyse des clés de jointure
print("Analyse des clés de jointure :")
print("-----------------------------")
print("df_erp['product_id'] :")
print("Nombre total de lignes :", len(df_erp))
print("Nombre de valeurs uniques :", df_erp['product_id'].nunique())
print("Type de données :", df_erp['product_id'].dtype)

print("\ndf_liaison['product_id'] :")
print("Nombre total de lignes :", len(df_liaison))
print("Nombre de valeurs uniques :", df_liaison['product_id'].nunique())
print("Type de données :", df_liaison['product_id'].dtype)

# Vérifier les valeurs qui ne correspondent pas
print("\nValeurs dans df_erp mais pas dans df_liaison :")
print(set(df_erp['product_id']) - set(df_liaison['product_id']))

print("\nValeurs dans df_liaison mais pas dans df_erp :")
print(set(df_liaison['product_id']) - set(df_erp['product_id']))

# Vérifier s'il y a des différences de format (espaces, casse, etc.)
print("\nVérification des formats potentiellement problématiques :")
print("Valeurs avec espaces dans df_erp:", df_erp[df_erp['product_id'].astype(str).str.contains(' ')]['product_id'].head())
print("Valeurs avec espaces dans df_liaison:", df_liaison[df_liaison['product_id'].astype(str).str.contains(' ')]['product_id'].head())

**Aucun soucis** entre ces deux fichiers, on peux directement se lancer dans la jointure

In [ ]:
#Fusion des fichiers df_erp et df_liaison
df_erp_liaison = pd.merge(df_erp, df_liaison, how="outer", on="product_id", indicator=True)

# Afficher les premières lignes du nouveau dataframe
df_erp_liaison.head()

In [ ]:
#Y a t-il des lignes ne "matchant" entre les 2 fichiers?
df_erp_liaison[df_erp_liaison["_merge"] != "both"]

Toutes les lignes **matchent** entre elles.

In [ ]:
#On peut supprimer la colonne _merge
df_erp_liaison.drop("_merge", axis=1, inplace=True)

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 3.2 - Jonction du fichier df_erp_liaison et df_web</h3>
</div>

In [ ]:
#Gestion des NaN et suppression des espaces inutiles 
df_erp_liaison['id_web'] = df_erp_liaison['id_web'].astype(str).str.strip().replace("nan", pd.NA)
df_web['sku'] = df_web['sku'].astype(str).str.strip().replace("nan", pd.NA)

#Vérifier le type de données des colonnes de jointure
print("Type de données dans df_erp_liaison['id_web']:", df_erp_liaison['id_web'].dtype)
print("Type de données dans df_web['sku']:", df_web['sku'].dtype)

In [ ]:
# Analyse des clés de jointure
print("Analyse des clés de jointure :")
print("-----------------------------")
print("df_erp_liaison['id_web'] :")
print("Nombre total de lignes :", len(df_erp_liaison))
print("Nombre de valeurs uniques :", df_erp_liaison['id_web'].nunique())
print("Type de données :", df_erp_liaison['id_web'].dtype)

print("\ndf_web['sku'] :")
print("Nombre total de lignes :", len(df_web))
print("Nombre de valeurs uniques :", df_web['sku'].nunique())
print("Type de données :", df_web['sku'].dtype)

# Vérifier les valeurs qui ne correspondent pas
print("\nValeurs dans df_erp_liaison mais pas dans df_web :")
print(set(df_erp_liaison['id_web']) - set(df_web['sku']))

print("\nValeurs dans df_web mais pas dans df_erp_liaison :")
print(set(df_web['sku']) - set(df_erp_liaison['id_web']))

# Vérifier les valeurs nulles ou problématiques
print("\nVérification des valeurs problématiques :")
print("Valeurs nulles dans df_erp_liaison['id_web'] :", df_erp_liaison['id_web'].isnull().sum())
print("Valeurs nulles dans df_web['sku'] :", df_web['sku'].isnull().sum())

In [ ]:
# Fusion des fichiers df_erp_liaison et df_web
df_final = pd.merge(df_erp_liaison, df_web, how="outer", left_on="id_web", right_on="sku", indicator=True)

# Vérification après jointure
print("\nVérification après jointure :")
print(f"Nombre de lignes dans df_erp_liaison : {len(df_erp_liaison)}")
print(f"Nombre de lignes dans df_web : {len(df_web)}")
print(f"Nombre de lignes dans df_final : {len(df_final)}")
print(f"Nombre de lignes avec sku null après jointure : {df_final['sku'].isnull().sum()}")

# Afficher les premières lignes du nouveau dataframe
df_final.head()

Après jointure on obtient bien **827** lignes (les 825 lignes issues de df_erp_liaison + les 2 lignes 80000 et 80001 présentes en plus dans df_web).

In [ ]:
#Y a t-il des lignes ne "matchant" entre les 2 fichiers?
df_final[df_final["_merge"] != "both"]

On a donc **2** lignes en right_only, qui sont les sku **80000 et 80001** que nous avons mis à la place des sku NaN pour éviter de supprimer les fiches produits (celles dont les total_sales étaient initialement négatives). Toutes les autres sont en left_only, et donc sont présent dans df_erp_liaison mais pas dans df_web.

Les **111** restantes sont les sku nulls : **91** id_web identifiés précédement dans le fichier df_liaison + **20** lignes identifiées juste avant (présentes dans df_erp_liaison et absentes de df_web).

In [ ]:
#On peut supprimer la colonne _merge
df_final.drop("_merge", axis=1, inplace=True)

<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 4 - Analyse univariée des prix</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.1 - Exploration par la visualisation de données</h3>
</div>

In [ ]:
#Création d'une Boite à moustache de la répartition des prix grâce à Pandas

import matplotlib.pyplot as plt

# Création de la boîte à moustaches
df_final.boxplot(column="price")

# Ajout d'un titre
plt.title("Répartition des prix")

# Affichage du graphique
plt.show()

In [ ]:
#Autre méthode avec plotly express

import plotly.express as px

# Création de la boîte à moustaches interactive
fig = px.box(df_final, y="price", title="Répartition des prix")

# Affichage du graphique
fig.show()

**Interprétation** :

**Concentration des prix** : La majorité des vins et spiritueux se situent entre 14,5€(Q1) et 42€(Q3), ce qui montre une offre bien concentrée sur cette gamme de prix.

**Forte dispersion au-delà de 83€ (outliers)** : Beaucoup de points au-dessus de cette valeur montrent qu’il existe une gamme premium avec des prix élevés, bien qu'ils soient moins nombreux.

**Distribution asymétrique** : Le fait que les valeurs s’espacent de plus en plus entre 144€ et 225€ indique une concentration de produits haut de gamme sous 150€, mais quelques références très chères qui font grimper le max à 225€.

**Segment du marché** : La médiane à 24,3€ montre que les vins et spiritueux les plus vendus sont abordables, mais qu’une partie du marché est axée sur des produits haut de gamme.

In [ ]:
#Analyse des prix par catégorie de produits

#Calculer la moyenne, médiane et écart-type des prix pour chaque catégorie (ex. vins rouges, blancs, spiritueux).
# Regroupement des données par catégorie de produit
df_categorie = df_final.groupby("product_type")["price"].agg(["mean", "median", "std"]).reset_index()
# Renommer les colonnes pour plus de clarté
df_categorie.columns = ["product_type", "mean_price", "median_price", "std_price"]
# Afficher les résultats
print(df_categorie)

#Vérifier si certaines catégories ont une plus grande dispersion des prix que d’autres.
# Trier les catégories par écart-type décroissant (les plus dispersées en premier)
df_categorie_sorted = df_categorie.sort_values(by="std_price", ascending=False)
# Afficher les catégories avec la plus grande dispersion
print(df_categorie_sorted.head(5))  
#Visualiser la dispersion des prix par catégorie avec un boxplot
plt.figure(figsize=(12, 6))
df_final.boxplot(column="price", by="product_type", rot=45, grid=False)
plt.title("Répartition des prix par catégorie")
plt.suptitle("")  
plt.xlabel("Catégorie de produit")
plt.ylabel("Prix (€)")
plt.show()
#Avec Plotly
fig = px.box(df_final, x="product_type", y="price", title="Répartition des prix par catégorie")
fig.show()

**Interprétations** : 

Les **catégories les plus chères** : Cognac et Champagne, avec des prix moyens de 97,50€ et 69,63€ respectivement.
Les **catégories les plus dispersées** : Cognac et Champagne ont les écarts-types les plus élevés (+47€), ce qui signifie qu'il y a des bouteilles à prix très variés.
Le **gin est un cas particulier** : Prix fixe à 36€ sans aucune dispersion. Cela peut indiquer une unique référence ou une politique de prix uniforme.
Le cognac est cher mais moins dispersé, ce qui peut signifier que la gamme est plus stable et orientée premium.
L’analyse confirme que certains segments sont très dispersés (Champagne, Cognac, Vin), alors que d’autres sont stables (Gin, Huile d’olive, Whisky en partie).

**Points intéressants à explorer**

Pourquoi le gin a-t-il un prix unique ? Une seule référence vendue ou un prix fixé artificiellement ?

In [ ]:
#Observation des produits gin
df_gin = df_final[df_final["product_type"] == "Gin"][["product_id", "product_type", "post_title", "price"]]
print(df_gin)


Tout est bien normal

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.2 - Exploration par l'utisation de méthodes statistiques</h3>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.2.1 - Identification par le Z-index</h3>
</div>

Le **Z-Score** mesure combien une valeur est éloignée de la moyenne en termes d'écart-type.

In [ ]:
#Vérifions s'il y a des NaN qui pourraient poser problème dans la colonne price
print(df_final["price"].isna().sum()) 
df_final[df_final["price"].isna()]

In [ ]:
#Pour s'éviter des problème lors du calcul, on remplace les valeurs NaN par la moyenne
mean_price = df_final["price"].mean()
df_final.fillna({"price" : mean_price}, inplace=True)

In [ ]:
from scipy.stats import zscore

#Calculer la moyenne du prix
mean_price = df_final["price"].mean()
#Calculer l'écart-type du prix
std_price = df_final["price"].std()
#Calculer le Z-score
z_score_price = mean_price / std_price
df_final["z_score"] = zscore(df_final["price"])

# Affichage des résultats
print(f"Moyenne du prix : {mean_price}")
print(f"Écart-type du prix : {std_price}")
print(f"Z-score du prix : {z_score_price}")
df_final[["price", "z_score"]].head()

In [ ]:
#Quel est le seuil prix dont z-score est supérieur à 3?

# Filtrer les prix dont le Z-score est supérieur à 3
seuil_prix = df_final.loc[df_final["z_score"] > 3, "price"].min()

print(f"Le seuil de prix dont le Z-score est supérieur à 3 est : {seuil_prix}")

**Interprétations** :

La majorité des prix sont concentrés autour de 32,19€, mais il existe une **forte dispersion**.

Les prix au-dessus de 114€ sont considérés comme exceptionnellement élevés selon cette analyse.

Ces valeurs extrêmes peuvent correspondre à des produits **haut de gamme ou des erreurs de saisie**.

In [ ]:
#Analyse des outliers (valeurs extrêmes) plus en détail

#Afficher la liste complète des produits avec un Z-score > 3.
outliers = df_final[df_final["z_score"] > 3][["product_id", "post_title", "product_type", "price", "z_score"]]
print(outliers)
#Vérifier les catégories de ces produits (exemple : vins rares, spiritueux haut de gamme).
categories_outliers = outliers["product_type"].value_counts()
print(categories_outliers)

#Comparer ces prix avec ceux d’autres produits similaires pour voir si l’écart est logique.
# Afficher les statistiques des prix par catégorie
price_stats_by_category = df_final.groupby("product_type")["price"].agg(["mean", "median", "std", "max"]).reset_index()
print(price_stats_by_category)
# Comparer les prix des outliers avec la moyenne et médiane de leur catégorie
outliers_with_category_stats = outliers.merge(price_stats_by_category, on="product_type", suffixes=("_outlier", "_category"))
print(outliers_with_category_stats)


**Interprétations** : 

Majorité des outliers dans les vins (**9** occurrences) et le champagne (**3** occurrences).
Les vins et champagnes semblent avoir **la plus grande dispersion de prix**.
Les outliers de Champagne (191-225€) sont bien supérieurs à la moyenne (69.63€) et même à son max habituel (~135€). → Ces produits sont des **millésimes rares**.
Les vins à 115-217.5€ sont bien plus chers que la moyenne (29.29€) → Ce sont sûrement des **grands crus ou des millésimes rares**.

Le whisky et le cognac comptent aussi quelques valeurs extrêmes (**2** occurrences chacun).
Les Cognacs à 157-176€ sont bien au-dessus de la moyenne (97.50€), mais l’écart-type (48.15€) montre que des prix élevés sont fréquents.
Les whiskys à 114-122€ restent cohérents avec leur max observé (122€), ce qui signifie que les outliers en whisky ne sont pas "anormaux" mais plutôt du haut de gamme.

In [ ]:
#Analyse des prix par catégorie de produits

#Analyser si le seuil des valeurs extrêmes (112€) est valable pour toutes les catégories ou si certaines ont naturellement des prix plus élevés.
# Calcul du seuil de Z-score > 3 pour chaque catégorie
df_final["z_score_by_category"] = df_final.groupby("product_type")["price"].transform(lambda x: (x - x.mean()) / x.std())
# Trouver le seuil spécifique de chaque catégorie
seuils_par_categorie = df_final.groupby("product_type")["price"].quantile(0.99)  # Seuil du top 1% de chaque catégorie
print(seuils_par_categorie)

**Interprétations** : 

Le seuil des outliers (**114€**) est **pertinent** pour les vins et champagnes.
Mais pour le cognac et le whisky, ce seuil pourrait être un peu bas → Ils ont naturellement des prix élevés.
Le gin et l'huile d'olive ne dépassent jamais 50€, donc ils ne sont pas concernés par cette analyse.

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.2.2 - Identification par l'interval interquartile</h3>
</div>

In [ ]:
#Utilisation de la fonction describe de Pandas pour l'etude des mesures de dispersions
df_final["price"].describe()

In [ ]:
#Définissez un seuil pour les articles "outliers" en prix
Q1 = df_final["price"].quantile(0.25)
Q3 = df_final["price"].quantile(0.75)
IQR = Q3 - Q1

seuil_bas = Q1 - 1.5 * IQR
seuil_haut = Q3 + 1.5 * IQR

print(f"Seuil inférieur : {seuil_bas}")
print(f"Seuil supérieur : {seuil_haut}")

**Interprétation** :

L’IQR nous montre qu’environ **50%** des produits ont un prix compris entre 14.50€ et 41.90€.
Les prix supérieurs à 83€ sont considérés comme **anormalement élevés**.
Le seuil inférieur (-26.6€) est inutile car il n’existe pas de prix négatif.

In [ ]:
#Définissez le nombre d'articles et la proportion de l'ensemble du catalogue "outliers"

# Identification des outliers
outliers = df_final[(df_final["price"] < seuil_bas) | (df_final["price"] > seuil_haut)]

# Nombre d'articles outliers
nb_outliers = outliers.shape[0]

# Proportion des outliers dans l'ensemble du catalogue
proportion_outliers = (nb_outliers / df_final.shape[0]) * 100

print(f"Nombre d'articles outliers : {nb_outliers}")
print(f"Proportion des outliers : {proportion_outliers:.2f}%")

**Interprétation** :

Seuls **4.35%** des produits ont un prix anormalement élevé.
Cela signifie que la plupart des produits ont un prix cohérent.

**Comparaison avec l’analyse par Z-score**

L’analyse par Z-score (étape précédente) considérait les prix au-dessus de 114€ comme des outliers.
Avec l’IQR, le seuil d’outlier est plus bas (83€) → **Cette méthode est donc plus stricte et considère plus de produits comme des valeurs extrêmes**.

In [ ]:
#Selon vous, ces outliers sont-ils justifiés ? Comment le démontrer si cela est possible ?

# 1. Histogramme des prix outliers
plt.figure(figsize=(10,6))
sns.histplot(outliers['price'], kde=True, bins=30, color='blue')
plt.title('Distribution des prix')
plt.xlabel('Prix')
plt.ylabel('Fréquence')
plt.show()

# 2. Affichage des outliers
display(outliers)

# 3. Occurence de chaque type de produits
occurrences_product_type = outliers["product_type"].value_counts()
print(occurrences_product_type)

**Interprétations** : 

Parmi ces outliers, **19** sont des vins (dont beaucoup sont des **grands cru** ou **1er cru**), **6** sont des champagnes, **4** des cognac et **3** des whisky.
Ces prix paraissent donc parfaitement justifiés dans un domaine (vins et spriritueux) où le millésime, le cépagne, l'appellation, etc, peuvent énormement faire varier les prix de vente.

On peut cependant ce demander si ces prix sont réellement justifiés, si les clients valident ces prix et de facto les achètent.

In [ ]:
# Sélection des articles outliers
outliers = df_final[df_final['z_score'].abs() > 3]

# Moyenne des ventes pour les outliers
mean_sales_outliers = outliers['total_sales'].mean()

# Moyenne des ventes pour les autres articles
mean_sales_non_outliers = df_final[df_final['z_score'].abs() <= 3]['total_sales'].mean()

print(f"Moyenne des ventes des outliers : {mean_sales_outliers}")
print(f"Moyenne des ventes des autres articles : {mean_sales_non_outliers}")

**D'après ces résultats** :

Les articles outliers ont une moyenne de ventes plus faible (**4.125**) que les autres articles (**8.14**).
Cela suggère que ces produits sont moins populaires ou que leur prix élevé limite les ventes.
Cependant, 4.125 ventes en moyenne n'est pas nulle, donc certains outliers se vendent tout de même.
**Ce qui confirme que ces outliers sont justifiés**.

<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 5 - Analyse univariée du CA, des quantités vendues, des stocks et de la marge ainsi qu'une analyse multivariée  </h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.1 - Analyse des ventes en CA</h3>
</div>

In [ ]:
##############################
# Calculer le CA su site web #
##############################

#Créez une colonne calculant le CA par article
df_final["ca_par_article"] = df_final["price"] * df_final["total_sales"]

#Calculez la somme de la colonne "ca_par_article"
ca_total = df_final["ca_par_article"].sum()
print(f"Chiffre d'affaires total du site : {ca_total}")

#Ce résultat correspond au chiffre d'affaires du site web

In [ ]:
###############################
# Palmares des articles en CA #
###############################

#Effectuer le tri dans l'ordre décroissant du CA du dataset df_final
df_top_ca = df_final.sort_values(by="ca_par_article", ascending=False)

#Réinitialiser l'index du dataset par un reset_index
df_top_ca = df_top_ca.reset_index(drop=True)

#Afficher les 20 premier articles en CA
df_top_ca.head(20)

#Graphique en barre des 20 premiers articles avec plotly express
fig = px.bar(df_top_ca.head(20), 
             x="post_title", 
             y="ca_par_article", 
             title="Top 20 des articles en chiffre d'affaires", 
             labels={"post_title": "Article", "ca_par_article": "Chiffre d'affaires"},
             text_auto=True)
fig.show()

Le **top 20** des articles montre les références les plus performantes :

Le Champagne Egly-Ouriet Grand Cru Millésimé 2008 est de loin le best-seller (**2475€**).
Beaucoup de Champagnes & vins prestigieux dans le top 20 → Cela suggère que le site cible une **clientèle haut de gamme**.
Certains articles ont un CA proche (entre 500€ et 800€), ce qui montre une **diversité de produits performants**.

In [ ]:
#Voyons si les articles du top CA sont vendus en grandes quantités ou s’ils ont un prix unitaire élevé

# Ajout d'une colonne CA par article
df_final["ca_par_article"] = df_final["price"] * df_final["total_sales"]

# Sélection du top 20 en CA
top_20_ca = df_final.groupby("product_type").agg(
    {"ca_par_article": "sum", "total_sales": "sum"}
).sort_values("ca_par_article", ascending=False).head(20)

# Ajout du prix unitaire moyen
top_20_ca["prix_unitaire_moyen"] = top_20_ca["ca_par_article"] / top_20_ca["total_sales"]

# Affichage des résultats
print(top_20_ca)

# Visualisation du CA vs Quantités vendues
fig = px.bar(top_20_ca, 
             x=top_20_ca.index, 
             y=["ca_par_article", "total_sales"], 
             barmode="group",
             title="Comparaison CA vs Quantités Vendues pour les 20 Meilleurs Articles",
             labels={"value": "Valeur", "variable": "Mesure"},
             text_auto=True)

fig.update_xaxes(tickangle=45)
fig.show()

Le Vin domine à la fois en chiffre d’affaires (**123162 €**) et en quantité vendue (**5448 unités**).
Le Champagne génère un CA élevé (**12 928 €**) mais avec peu de ventes (**169 unités**) ➝ Prix moyen élevé (76.50 €).
Le Cognac et le Whisky ont un prix moyen élevé (90.57 € pour le Cognac, 60.13 € pour le Whisky), mais les ventes restent faibles (35 et 48 unités respectivement).
L’Huile d’olive a un faible CA (497.7 €) et peu de ventes (22 unités) ➝ Produit marginal.

**Conclusion** :

Les vins génèrent le plus de CA grâce aux volumes de vente élevés, malgré un prix moyen faible (22.61 €).
Les spiritueux (Cognac, Whisky) et le Champagne ont un CA plus basé sur leur prix que sur les quantités vendues.

Le **principe de Pareto** dit que 80% des effets proviennent de 20% des causes.
En entreprise, 80% du chiffre d’affaires provient souvent de 20% des produits ou 20% des clients.

In [ ]:
#############################
# Calculer le 20 / 80 en CA #
#############################

#Créer une colonne calculant la part du CA de la ligne dans le dataset
df_top_ca["part_ca"] = df_top_ca["ca_par_article"] / df_top_ca["ca_par_article"].sum()

#Créer une colonne réalisant la somme cumulative de la colonne précedemment créée
df_top_ca["cum_sum_ca"] = df_top_ca["part_ca"].cumsum()

#Grâce au deux colonnes créées précedemment, calculer le nombre d'articles représentant 80% du CA
nb_articles_80 = (df_top_ca["cum_sum_ca"] <= 0.80).sum()
print(f"Nombre d'articles représentant 80% du CA : {nb_articles_80}")

#Afficher la proportion que représentent ce groupe d'articles dans le catalogue entier du site web
total_articles = df_final.shape[0]
proportion_articles = nb_articles_80 / total_articles * 100
print(f"Ces articles représentent {proportion_articles:.2f}% du catalogue total.")


En général, on observe que 20% des articles génèrent environ 80% du CA (principe de Pareto). Ici, non.

Cependant, 80% du CA est généré par seulement 433 articles, soit 52.8% du catalogue.
Cela confirme que **près de la moitié des produits contribuent peu aux revenus**, tandis qu’une minorité domine.

**Stratégie possible** : focaliser les efforts marketing et stocks sur ces 433 articles pour maximiser la rentabilité.

In [ ]:
#Analysons les caractéristiques communes des best-sellers

# Sélection des articles représentant 80% du CA
df_final["part_CA"] = df_final["ca_par_article"] / df_final["ca_par_article"].sum()
df_final["cumul_CA"] = df_final["part_CA"].cumsum()
best_sellers = df_final[df_final["cumul_CA"] <= 0.8]

# Analyse des caractéristiques : Répartition par Type de produit
type_counts = best_sellers["product_type"].value_counts()
fig1 = px.pie(values=type_counts.values, 
              names=type_counts.index, 
              title="Répartition des Best-Sellers par Type de Produit")
fig1.show()

# Répartition par Gamme de prix
def categoriser_prix(prix):
    if prix < 50:
        return "-50€"
    elif 50 <= prix < 100:
        return "50-100€"
    elif 100 <= prix < 200:
        return "100-200€"
    else:
        return "+200€"

best_sellers = best_sellers.copy()
best_sellers["gamme_prix"] = best_sellers["price"].apply(categoriser_prix)
prix_counts = best_sellers["gamme_prix"].value_counts()
fig2 = px.bar(x=prix_counts.index, 
              y=prix_counts.values, 
              title="Répartition des Best-Sellers par Gamme de Prix",
              labels={"x": "Gamme de Prix", "y": "Nombre d'Articles"})
fig2.show()

**Conclusion** :

Les best-sellers sont des vins abordables (**<50€**), ce qui confirme que le volume des ventes est plus important que le prix unitaire.
Le Champagne et les spiritueux vendent moins en quantité, mais contribuent au CA grâce à des prix élevés.

**Conclusion & insights business**

Le segment clé semble être **les vins entre 15€ et 30€**, car la médiane est à 21.80€.
Quelques vins chers (**>100€**) existent, mais ils sont minoritaires dans les ventes.
L’offre est bien diversifiée avec des produits pour tous les budgets.

Pour maximiser les ventes, il pourrait être intéressant d’analyser encore plus les performances des vins entre 15-30€ pour voir ce qui marche le mieux 
(marque, origine, cépage, etc.).

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.2 - Analyse des ventes en Quantités</h3>
</div>

In [ ]:
#Analysons la distribution des prix des vins

# Filtrer uniquement les vins
df_vins = df_final[df_final["product_type"] == "Vin"]

# Histogramme des prix
plt.figure(figsize=(10, 5))
sns.histplot(df_vins["price"], bins=20, kde=True)
plt.xlabel("Prix unitaire (€)")
plt.ylabel("Nombre de ventes")
plt.title("Distribution des prix des vins vendus")
plt.show()

# Statistiques clés
prix_mean = df_vins["price"].mean()
prix_median = df_vins["price"].median()
prix_std = df_vins["price"].std()

print("\nPrix minimum : {:.2f}€".format(df_vins['price'].min()))
print("Prix maximum : {:.2f}€".format(df_vins['price'].max()))
print(f"Prix moyen des vins vendus : {prix_mean:.2f}€")
print(f"Médiane des prix des vins vendus : {prix_median:.2f}€")
print(f"Écart-type des prix des vins vendus : {prix_std:.2f}€")

**Interprétation** :

La médiane (**21.95€**) est inférieure à la moyenne (**29.29€**) → La distribution des prix est asymétrique à droite : il y a quelques vins très chers qui tirent la moyenne vers le haut.
Un écart-type de 23.42€ signifie une forte dispersion des prix → Certains vins sont très abordables, d'autres sont nettement plus chers.

In [ ]:
#####################################
# Palmares des articles en quantité #
#####################################

#Effectuer le tri dans l'ordre décroissant de quantités vendues du dataset df_merge
df_top_quantite = df_final.sort_values(by="total_sales", ascending=False)

#Réinitialiser l'index du dataset par un reset_index
df_top_quantite = df_top_quantite.reset_index(drop=True)

#Afficher les 20 premier articles en quantité
df_top_quantite.head(20)

#Graphique en barre des 20 premiers articles avec plotly express
fig = px.bar(df_top_quantite.head(20), x="post_title", y="total_sales", 
             title="Top 20 des articles en quantité vendue", 
             labels={"post_title": "Article", "total_sales": "Quantité vendue"},
             text="total_sales")
fig.show()

Cela peut être utile pour **optimiser les stocks et les stratégies de mise en avant**.

**Interprétation** :

Il y a un fort écart entre les premières positions et le reste du classement → **Quelques vins dominent les ventes**.
Les rosés et les blancs semblent bien représentés dans les meilleures ventes.
L’effet millésime joue un rôle important : la majorité des vins du top 20 sont récents (2017-2019).

In [ ]:
#Comparer ces vins avec leurs prix : les vins les plus vendus sont-ils dans la fourchette de prix basse, moyenne ou haute ?

# Calcul des quartiles des prix
Q1 = df_final["price"].quantile(0.25)
Q3 = df_final["price"].quantile(0.75)

# Définir les catégories de prix
def categorie_prix(prix):
    if prix < Q1:
        return "Bas"
    elif prix <= Q3:
        return "Moyen"
    else:
        return "Élevé"

# Appliquer la catégorisation au DataFrame complet
df_final["categorie_prix"] = df_final["price"].apply(categorie_prix)

# Sélectionner les 20 vins les plus vendus
df_top_20 = df_final[df_final["product_type"] == "Vin"].sort_values(by="total_sales", ascending=False).head(20)

# Vérifier la répartition des catégories de prix dans le top 20
repartition_top_20 = df_top_20["categorie_prix"].value_counts()

# Affichage des résultats
print("Répartition des catégories de prix dans le top 20 des ventes :")
print(repartition_top_20)

# Visualisation avec un graphique à barres
plt.figure(figsize=(8, 5))
sns.barplot(x=repartition_top_20.index, y=repartition_top_20.values, hue=repartition_top_20.index, palette="viridis", legend=False)
plt.xlabel("Catégorie de prix")
plt.ylabel("Nombre de vins")
plt.title("Répartition des catégories de prix parmi les 20 vins les plus vendus")
plt.show()

**Interprétations** : 

Les vins les plus populaires sont majoritairement des vins à bas prix.

**75%** des 20 vins les plus vendus appartiennent à la catégorie "**Bas prix**".
Cela indique que les clients privilégient des **bouteilles abordables** pour leurs achats.
Les vins de prix moyen restent achetés mais en quantité moindre.

Seuls 5 vins sur 20 (**25%**) appartiennent à la **gamme moyenne**.
Cela suggère que certains consommateurs sont prêts à payer plus, mais cela reste une minorité.

Aucun vin de catégorie élevée ne se trouve dans le top 20 des ventes.
Cela peut s'expliquer par un prix trop élevé pour le grand public ou une demande plus spécifique et limitée.

In [ ]:
#############################
# Calculer le 20 / 80 en CA #
#############################

#Créer une colonne calculant la part en quantité de la ligne dans le dataset
df_final["part_quantite"] = df_final["total_sales"] / df_final["total_sales"].sum()

#Créer une colonne réalisant la somme cumulative de la colonne précedemment créée
df_final = df_final.sort_values(by="total_sales", ascending=False)  # Tri décroissant
df_final["cumsum_part_quantite"] = df_final["part_quantite"].cumsum()

#Grâce au deux colonnes créées précedemment, calculer le nombre d'articles représentant 80% des ventes en quantité
nb_articles_80 = (df_final["cumsum_part_quantite"] <= 0.8).sum()
print(f"Nombre d'articles représentant 80% des ventes : {nb_articles_80}")

#Afficher la proportion que représentent ce groupe d'articles dans le catalogue entier du site web
proportion_catalogue = nb_articles_80 / len(df_final) * 100
print(f"Proportion des articles générant 80% des ventes : {proportion_catalogue:.2f}%")


Si la proportion est faible (ex : 20% des articles génèrent 80% des ventes), alors on valide la loi Pareto 20/80. Ici, non.

**Interprétation** :

Plus de la moitié du catalogue est essentielle aux ventes → Il n’y a **pas une hyper-concentration** sur un très petit nombre de références.
Presque la moitié du catalogue a une contribution faible aux ventes en quantités → Cela pourrait indiquer des produits de niche ou des références qui mériteraient d’être mieux mises en avant.

**Recommandations** :

Si l’objectif est d’augmenter les ventes globales :

Miser sur les vins à bas prix, car ils constituent la majorité des ventes.
Promouvoir les vins de milieu de gamme (offres, packs, mises en avant) pour tenter d’élargir la clientèle.
    
Si l’objectif est d’augmenter le chiffre d’affaires :

Travailler sur la montée en gamme (suggestions d’achat, marketing autour des vins moyens et hauts de gamme).
Proposer des offres spéciales ou des packs découverte pour inciter les clients à tester des vins plus chers.

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.3 - Analyse des stocks</h3>
</div>

In [ ]:
######################################
# Calcule le nombre de mois de stock #
######################################

#Import de numpy 
import numpy as np

#Création de la colonne Rotation de stock
df_final["mois_stock"] = df_final["stock_quantity"] / df_final["total_sales"]

#Remplacement des "inf" par 0
df_final["mois_stock"] = df_final["mois_stock"].replace([np.inf, -np.inf], 0)
df_final["mois_stock"] = df_final["mois_stock"].fillna(0)

#Effectuer le tri dans l'ordre décroissant du nombre de mois de stock dans le dataset df_final
df_final = df_final.sort_values(by="mois_stock", ascending=False)

#Graphique en barre du flop 20 des produits qui ont le plus de mois de stock
fig = px.bar(df_final.head(20), x="post_title", y="mois_stock", 
             title="Flop 20 des produits avec le plus de stock restant",
             labels={"post_title": "Produit", "mois_stock": "Mois de stock"},
             text_auto=True)

fig.show()

**Constats clés** :

Ces produits se vendent très lentement, entraînant une **immobilisation du capital dans le stock**.
Une forte présence de champagnes parmi ces produits suggère une **faible rotation** de ce type d’articles dans le catalogue actuel.

**Produits avec un stock élevé** :
→ Ceux ayant plusieurs mois de stock risquent d’être immobilisés trop longtemps.

**Produits à écouler en priorité** :
→ Les produits avec rotation faible (stock stagnant) devraient être mis en promotion ou retirés du catalogue.

**Impacts sur la gestion des stocks** :

Ces produits doivent être écoulés rapidement via :
Promotions ciblées (réductions, ventes privées).
Mises en avant marketing (événements spéciaux, recommandations sur le site).
Possibilité de réduire les commandes fournisseurs pour ces articles à l’avenir.

In [ ]:
####################################
# Valorisation des stocks en euros #
####################################

#Création de la colonne Valorisation des stocks en euros
df_final["Valorisation_stock_euros"] = df_final["stock_quantity"] * df_final["purchase_price"]

#Calculer la somme de la colonne "Valorisation_stock_euros"
valorisation_totale = df_final["Valorisation_stock_euros"].sum()
print(f"Valorisation totale des stocks : {valorisation_totale:.2f} €")

In [ ]:
##############################################
# Valorisation du nombre de produit en stock #
##############################################

#Calculer la somme de la colonne stock quantity
total_produits_stock = df_final["stock_quantity"].sum()
print(f"Nombre total de produits en stock : {total_produits_stock}")

**Constats clés** :

Une immobilisation de près de **300K €** en valeur stockée représente un risque financier important.
Une **optimisation du stock** permettrait de réduire les coûts de stockage et d’améliorer la rentabilité.

**17 822 produits en stocks - 11; puisque nous avons remis les stocks des produits 4973 et 5700 (product_id), qui initialement étaient respectivement de -10 et -1, à 0.**

In [ ]:
#Plutôt que de seulement regarder les mois de stock, il serait intéressant d’identifier les produits immobilisant le plus de valeur financière.

# Trier les produits ayant la plus forte valorisation de stock
df_stock_valorise = df_final.sort_values(by="Valorisation_stock_euros", ascending=False)
# Afficher les 20 produits qui représentent le plus de valeur stockée
df_stock_valorise.head(20)

La majorité des produits ayant la plus grande valeur en stock sont des Champagnes.

Le **Coteaux Champenois Egly-Ouriet Ambonnay rouge** est le produit avec la plus forte valorisation en stock (98 unités à 191,30€).
D'autres produits comme le Champagne Gosset Célébris Vintage 2007 (138 unités à 135€) ou le Champagne Agrapart & fils l’Avizoise Extra Brut blanc de blanc Grand Cru 2012 (136 unités à 112€) représentent aussi des stocks très valorisés.

**Interprétations** :

Ces produits représentent une **immobilisation financière importante**, ce qui peut poser problème si leur rotation est faible.
Il est essentiel d'évaluer leur vitesse de vente pour éviter une sur-accumulation.

In [ ]:
#Comparer la valorisation du stock avec le chiffre d'affaires généré

# Calcul du ratio "Valorisation des stocks" / "Chiffre d'affaires généré"
df_final["total_sales_value"] = df_final["total_sales"] * df_final["price"]
df_final["ratio_stock_CA"] = df_final["Valorisation_stock_euros"] / df_final["total_sales_value"]
# Trier par ratio décroissant (produits où la valeur du stock est très élevée par rapport aux ventes)
df_ratio_stock_CA = df_final.sort_values(by="ratio_stock_CA", ascending=False)
# Afficher les 20 articles les plus problématiques
df_ratio_stock_CA.head(20)

Plusieurs produits ont un ratio "Inf" : ils n’ont **pas généré de ventes malgré un stock élevé**.
Ex : Cognac Normandin Mercier VFC, Champagne Egly-Ouriet Grand Cru Blanc de Noirs, Champagne Mailly Grand Cru Les Echansons 2007.

Certains produits ont un ratio supérieur à 10, ce qui signifie que la **valeur en stock est 10x supérieure au chiffre d’affaires généré**.
Ex : Champagne Gosset Grand Millésime 2006 (18.95x), Champagne Gosset Célébris Vintage 2007 (16.42x).

**Interprétations** :

Ces produits pourraient être mal positionnés en prix, peu attractifs pour les clients, ou leur demande est faible.
Il faut mettre en place des actions correctives pour accélérer leur écoulement.

Il y a un risque de **surstock** important sur certains produits.


**Recommandations** :

*Identifier les causes du faible écoulement des stocks* :

Ont-ils un problème de visibilité ? Vérifier s'ils sont bien mis en avant sur le site / en magasin.
Manque de demande ? Vérifier les tendances des ventes passées et les avis clients.

*Mettre en place des actions correctives* :

Promotions ciblées : Réduction des prix sur les produits avec un stock élevé.
Mise en avant marketing : Campagnes e-mailing, publicité sur les produits en surstock.
Packs & offres spéciales : Associer ces produits avec des best-sellers.

*Vérifier la saisonnalité* :

Certains de ces produits sont-ils des achats saisonniers ?
Ex : Si les ventes de Champagne augmentent en fin d’année, alors le surstock pourrait être temporaire.

*Optimiser la gestion des stocks* :
 
Diminuer les nouvelles commandes sur les produits à forte valorisation.
Augmenter les commandes sur les produits avec une forte rotation.

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.4 - Analyse du taux de marge</h3>
</div>

In [ ]:
############################
# Analyse du taux de marge #
############################

#Création de la colonne prix HT
df_final["prix_ht"] = df_final["price"] / 1.2  # Conversion du prix en HT
df_final["prix_achat_ht"] = df_final["purchase_price"] / 1.2

#Création de la colonne Taux de marge
df_final["taux_marge"] = (df_final["prix_ht"] - df_final["prix_achat_ht"]) / df_final["prix_achat_ht"]

#Afficher le prix minimum de la colonne "taux_marge"
print(f"Taux de marge minimum : {df_final['taux_marge'].min()}")
#Afficher le prix maximum de la colonne "taux_marge"
print(f"Taux de marge maximum : {df_final['taux_marge'].max()}")

**Interprétation** :

Un taux négatif indique des **ventes à perte**, ce qui est préoccupant.
Un taux maximum de 129.7% suggère des produits à **forte marge**, généralement des produits premium ou de niche.

In [ ]:
#Affichage des lignes avec un taux de marge inférieur à 0
df_perte = df_final[df_final["taux_marge"] < 0]
print(df_perte[["product_id", "post_title", "price", "purchase_price", "taux_marge"]])

**Problème** :

Le Champagne Egly-Ouriet Grand Cru Blanc de Noirs est vendu largement en dessous de son prix d’achat (-6,12x de perte !)
D’autres produits sont également vendus légèrement à perte (-0.6% à -16.9%).
D'autres encore sont vendus à perte du fait de leur prix négatifs.

**Causes possibles** :

Erreur de pricing ? Vérifier si les prix ont été mal renseignés.
Politique de déstockage ? Certains produits sont peut-être en promotion pour écouler les stocks.
Problème de calcul ? Peut-être une erreur dans le fichier des prix d'achat ?

On retrouve également dans cette liste les **3 produits que nous avions identifiés lors du nettoyage des données, produits dont les prix de ventes sont négatifs**.
Il est interdit de vendre à perte, il faut voir avec un référent métier d'où proviennent ces prix négatifs et les corriger en conséquence.

In [ ]:
#Identifier les produits avec une marge égale à 0

#Filtrer les produits avec une marge de 0
df_marge_nulle = df_final[df_final["taux_marge"] == 0]

#Afficher les produits concernés
df_marge_nulle[["product_id", "post_title", "price", "purchase_price", "taux_marge"]]

Aucun produit vendu avec une marge égale à 0.
Il n’y a donc pas de produits vendus "au prix coûtant", ce qui est un bon indicateur de gestion des prix.

In [ ]:
#création d'un dataframe avec les taux positifs
df_marge_positive = df_final[df_final["taux_marge"] > 0]

#Afficher le prix minimum de la colonne "taux_marge"
print("Taux de marge minimum :", df_marge_positive["taux_marge"].min())
#Afficher le prix maximum de la colonne "taux_marge"
print("Taux de marge maximum :", df_marge_positive["taux_marge"].max())

#Un taux de marge minimum proche de 0 peut indiquer des articles peu rentables.
#Un taux de marge maximum élevé peut révéler des produits premium ou à forte marge.

In [ ]:
#création d'un dataframe avec le taux de marge moyen par type de produit
df_marge_par_type = df_final.groupby("product_type")["taux_marge"].mean().reset_index()

#Affichage dans un graphique du taux de marge par type de produit
fig = px.bar(df_marge_par_type, x="product_type", y="taux_marge", 
             title="Taux de marge moyen par type de produit",
             labels={"taux_marge": "Taux de Marge", "product_type": "Type de Produit"},
             color="taux_marge", color_continuous_scale="Blues")

fig.show()

**Interprétations** :

Cognac (**1.1878**) et Whisky (**1.1809**)
→ Ce sont les produits les plus rentables avec une marge de plus de 100 %, soit un prix de vente plus du double du coût d'achat. L'entreprise réalise donc une forte plus-value sur ces alcools, ce qui peut indiquer une forte demande, une image de marque premium, ou un positionnement haut de gamme.

Gin (**1.0979**)
→ Une marge élevée également, proche des 110 %, ce qui reste très intéressant. Cela montre que le gin est aussi un produit rentable, mais légèrement moins que le cognac et le whisky.

Vin (**0.9379**)
→ Une marge inférieure aux spiritueux, mais proche de 100 %, ce qui signifie que le prix de vente est quasiment le double du coût d'achat. Cela reste un produit rentable, mais avec une concurrence peut-être plus forte ou des prix de marché plus encadrés.

Champagne (**0.6253**) et Huile d’olive (**0.6007**)
→ Ces produits ont les marges les plus faibles (environ 60 %). Cela peut s’expliquer par plusieurs facteurs :

Une forte concurrence sur le marché qui limite les prix de vente.
Un coût d’achat relativement élevé, réduisant la marge bénéficiaire.
Une politique de prix plus agressive pour attirer les clients avec des produits populaires.

**Analyse globale** :

L’entreprise génère ses plus fortes marges sur les spiritueux (Cognac, Whisky, Gin).
Le vin reste un produit rentable mais légèrement en retrait.
Le champagne et l’huile d’olive offrent des marges plus faibles, ce qui peut refléter une nécessité de volume de ventes plus important pour être rentables.
Une stratégie intéressante pourrait être d’optimiser les prix ou de réduire les coûts des produits à faible marge pour améliorer la rentabilité globale.

In [ ]:
#Vérifier les produits peu rentables mais populaires

# Définir un seuil de rentabilité faible (par exemple, marge < 20%)
seuil_marge_faible = 0.20

# Trier les produits ayant une faible marge mais un volume de ventes élevé
df_faible_marge_populaire = df_final[(df_final["taux_marge"] < seuil_marge_faible) & 
                                     (df_final["total_sales"] > df_final["total_sales"].quantile(0.75))]  
# 75e percentile = produits parmi les 25% les plus vendus

# Trier par volume de ventes décroissant
df_faible_marge_populaire = df_faible_marge_populaire.sort_values(by="total_sales", ascending=False)

# Afficher les 20 produits concernés
df_faible_marge_populaire[["product_id", "post_title", "price", "total_sales", "taux_marge"]].head(20)

**Interprétations** : 

Avec le seuil fixé à 0.20 de taux de marge, aucun produit ne se distingue comme étant fortement vendu tout en ayant une faible rentabilité.

Cela signifie qu’il n’y a **pas d’effet "volume compensant une faible marge"**, ce qui peut être une bonne ou une mauvaise chose selon la stratégie commerciale :
Bonne nouvelle : **Pas de produits sous-évalués** qui génèrent du chiffre d’affaires sans bénéfice.
Moins bonne nouvelle : Une stratégie prix pourrait être envisagée pour booster les ventes de certains produits avec une marge un peu plus faible.

In [ ]:
#Comparer les taux de marge avec le volume des ventes

fig = px.scatter(df_final[df_final["total_sales"].notna()], 
                 x="taux_marge", y="total_sales", 
                 size="total_sales", color="taux_marge",
                 title="Relation entre le taux de marge et le volume des ventes",
                 labels={"taux_marge": "Taux de marge", "total_sales": "Volume des ventes"},
                 hover_data=["post_title", "price"])

fig.update_layout(xaxis=dict(range=[0, 1.5]))  # Fixe l'axe des abscisses

fig.show()

**Interprétations** : 

Le graphique montre une forte concentration des ventes pour des **taux de marge situés entre 0.8 et 1.2**.
Peu de produits ont une marge inférieure à 0.6, ce qui confirme que l’entreprise privilégie des marges confortables.
**Aucune tendance évidente** ne montre qu’un taux de marge plus bas entraînerait plus de ventes, ce qui indique que le prix n’est pas le seul facteur déterminant la demande.

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.5 - Analyse des correlations entre les variables stock, sales et price</h3>
</div>

In [ ]:
############################
# Analyse des correlations #
############################

#Importation de Seaborn
import seaborn as sns
import matplotlib.pyplot as plt

#Création d'un heatmap de correlation avec les variables stock, sales et price
#Création de la Matrice de Corrélation
correlation_matrix = df_final[["stock_quantity", "total_sales", "price"]].corr()

#On peut également créer un mask pour n'afficher qu'une demi heatmap
#Création d'un mask pour afficher une demi-heatmap
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

#Affichage du Heatmap avec Seaborn
plt.figure(figsize=(8,6))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5, mask=mask)
plt.title("Heatmap des Corrélations entre Stock, Ventes et Prix")
plt.show()

Que peut-on conclure des correlations ?


*Corrélation entre Stock et Ventes* (+0.44)

Une **corrélation modérément positive** entre le stock disponible et les ventes signifie que, globalement, plus un produit est en stock, plus il est vendu.
Cela peut être dû au fait que les produits les plus populaires sont mieux approvisionnés, ou que la disponibilité influence directement la demande.

*Corrélation entre Prix et Ventes* (-0.51)

Une **corrélation négative assez marquée** signifie que plus un produit est cher, moins il se vend.
Cela est conforme aux attentes : les produits coûteux attirent moins d’acheteurs que les produits à prix bas ou modéré.
Attention : Cette corrélation ne prouve pas une causalité directe (des produits chers peuvent aussi être de niche et avoir une demande limitée).

*Corrélation entre Stock et Prix* (-0.09)

**Corrélation très faible**, donc pas de lien significatif entre le niveau de stock et le prix des produits.
Cela signifie que la gestion des stocks ne dépend pas directement du prix : les produits chers ne sont pas forcément stockés en moindre quantité et inversement.

In [ ]:
#Analyse des ruptures de stock

# Définition des seuils
stock_seuil = 5  # Considérer un stock faible si <= 5 unités
vente_seuil = 1  # Un produit est en demande s'il a au moins une vente

# Filtrer les produits avec un stock faible mais des ventes présentes
ruptures_stock = df_final[(df_final["stock_quantity"] <= stock_seuil) & (df_final["total_sales"] > vente_seuil)]

# Afficher les produits concernés
if ruptures_stock.empty:
    print("Aucune rupture de stock détectée pour les produits en demande.")
else:
    print("Produits en risque de rupture de stock :")
    display(ruptures_stock[["post_title", "stock_quantity", "total_sales", "price"]])

**Interprétations** : 

*Produits totalement en rupture de stock (stock = 0)*

Ces produits ont des ventes élevées malgré l'absence de stock.

Exemples :
François Baur Pinot Noir Schlittweg 2017 (22 ventes, 0 stock, 12.70€)
Champagne Egly-Ouriet Grand Cru Millésimé 2008 (11 ventes, 0 stock, 225.00€)
Parcé Frères IGP Côtes Catalanes Hommage à Fernand 2018 (15 ventes, 0 stock, 8.90€)
Triennes IGP Méditerranée Rosé 2019 (16 ventes, 0 stock, 9.30€)

Problème : Une rupture de stock sur des produits populaires peut entraîner un manque à gagner.

Recommandations :
Vérifier si ces produits peuvent être réapprovisionnés rapidement.
Analyser s'il faut augmenter les quantités commandées pour éviter ces situations (analyse à faire sur plusieurs mois).
Identifier si des alternatives similaires sont disponibles pour satisfaire la demande.

*Produits en stock critique (1-5 unités restantes)*

Ces produits sont encore disponibles mais risquent d'être en rupture prochainement.

Exemples :
David Duband Morey-Saint-Denis 2017 (4 en stock, 2 ventes, 48.70€)
Lucien Boillot Puligny-Montrachet 1er Cru (4 en stock, 2 ventes, 83.70€)
Cognac Frapin VSOP (5 en stock, 3 ventes, 62.50€)
Marc Colin Et Fils Chassagne-Montrachet (5 en stock, 4 ventes, 68.30€)

Problème : Ces produits sont à surveiller car ils sont en forte demande et peuvent être en rupture sous peu.

Recommandations :
Augmenter la fréquence de suivi des stocks sur ces références.
Analyser la vitesse de vente pour ajuster les réassorts (idem sur plusieurs mois, plus pertinent que sur un seul).
Vérifier si des promotions pourraient accélérer la vente avant qu'une nouvelle commande arrive.

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.6 - Mettre à disposition la nouvelle table sur un fichier Excel</h3>
</div>

#Mettre le dataset df_merge sur un fichier Excel
df_final.to_excel("C:/Users/user-aidicom/Desktop/Recherche emploi/2024/cours/OpenClassrooms/Projets/Projet n°6 Optimisez la gestion des données d'une boutique avec R ou Python/Fcihiers de travail.xlsx", index=False)


#Cette étape peut-être utile pour partager le résultat du dataset obtenu pour le partager avec les équipes.  
